# IGLOO Playground — GB1 additive / Linear (standalone / Colab edition)

An end-to-end exploratory-analysis playground for **GB1 additive, Linear interpolation**. This copy vendors its own `src/igloo` package, data, and two pre-trained runs (the best model-performance cell, and a shuffled-label "complete random" negative control) so it runs with **no dependency on the original repo or cluster** — just this folder.

Every heavy black-box call in the production pipeline is unpacked here into separate cells so you can inspect **tensor shapes and raw values at each stage**: embeddings -> attention pooling -> integrated gradients -> all 7 evaluation metrics. Goals of this notebook:
- See the intermediate computation behind every metric (NDCG, AUROC, AP@R, per-variant NDCG, sign accuracy, category separation, mutation NDCG), not just the final number.
- Understand *why* some metrics can't distinguish real signal from a random-shuffled-label baseline, by directly comparing the best model against that negative control at every stage of the pipeline.
- Give you room to prototype new metrics and check whether they're more discriminative.

**Before running anything: Runtime -> Change runtime type -> GPU.** The embedding and integrated-gradients cells are much faster with one, though everything works on CPU too (just slower).

## 0. Setup

If you're on Colab and just unzipped this into `/content/`, make sure your working directory is the folder containing this notebook (the one with `igloo_playground_utils.py`, `src/`, `data/`, `results/`) before running the cells below, e.g.:
```python
# only needed once, if you're not already cd'd into the unzipped folder
# !unzip -o colab_standalone.zip
# %cd colab_standalone
```

In [ ]:
# Colab-only: install packages this notebook needs that aren't preinstalled.
# (torch/numpy/pandas/matplotlib/scipy/scikit-learn are already on Colab.)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q esm safetensors pyyaml seaborn biopython rich typer
print('Running on Colab:', IN_COLAB)

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

def _find_notebook_dir():
    # 1) already in the right place; 2) one level down (common after unzipping
    # to /content/ leaves files under /content/colab_standalone/); 3) last resort,
    # search a couple levels under cwd.
    candidates = [Path.cwd(), Path.cwd() / 'colab_standalone']
    for c in candidates:
        if (c / 'igloo_playground_utils.py').exists():
            return c.resolve()
    for depth_glob in ('*/igloo_playground_utils.py', '*/*/igloo_playground_utils.py'):
        hits = list(Path.cwd().glob(depth_glob))
        if hits:
            return hits[0].parent.resolve()
    raise FileNotFoundError(
        'Could not find igloo_playground_utils.py under ' + str(Path.cwd()) + '. '
        "Make sure you've unzipped colab_standalone.zip and either %cd'd into "
        'the folder that directly contains igloo_playground_utils.py / src/ / '
        'data/ / results/, or left it one level below your current directory.'
    )

NOTEBOOK_DIR = _find_notebook_dir()
sys.path.insert(0, str(NOTEBOOK_DIR))
import igloo_playground_utils as ig

PROJECT_ROOT = ig.PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / 'data' / 'synthetic'
print('notebook dir  :', NOTEBOOK_DIR)
print('project root  :', PROJECT_ROOT)
print('device        :', ig.DEVICE)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU detected -- Runtime > Change runtime type > GPU for a much faster experience.')

## 1. Load the metrics summary and pick a run

`GB1_additive_signal_families_vs_model_perf_raw_linear.csv` has all 60 (noise, seed) cells x 7 metrics, precomputed on the original cluster. First: which cell has the **best model performance** (Spearman rho between predicted and true fitness)? That's this bundle's primary focus. We also show the best cell **per metric** for reference — but note only noise=0.0/seed_1 (this bundle's one shipped "best" run) has a checkpoint + IG results included, alongside one shuffled-label "complete random" run (random_baseline/seed_0) used throughout as a negative control.

In [ ]:
metrics_df = pd.read_csv(DATA_DIR / 'GB1_additive_signal_families_vs_model_perf_raw_linear.csv')
baseline_df = pd.read_csv(DATA_DIR / 'GB1_additive_random_baseline_linear.csv')
print(f'{len(metrics_df)} (noise, seed) cells x {metrics_df.shape[1]} columns')
metrics_df.sort_values('x_value', ascending=False).head(8)

In [ ]:
METRIC_KEYS = ['ndcg', 'auroc', 'ap_graded', 'pv_ndcg_mean', 'sign_acc', 'cat_sep', 'mutation_ndcg_mean_per_seq']
METRIC_LABELS = {
    'ndcg': 'Global NDCG', 'auroc': 'AUROC', 'ap_graded': 'AP@R (graded)',
    'pv_ndcg_mean': 'Per-variant NDCG', 'sign_acc': 'Sign accuracy',
    'cat_sep': 'Category separation', 'mutation_ndcg_mean_per_seq': 'Mutation-level NDCG',
}

print('=== best (noise, seed) per metric ===')
best_per_metric = []
for m_key in METRIC_KEYS:
    row = metrics_df.loc[metrics_df[m_key].idxmax()]
    best_per_metric.append({'metric': m_key, 'best_value': row[m_key], 'noise': row.noise, 'seed': row.seed, 'model_perf_x': row.x_value})
pd.DataFrame(best_per_metric)

### Selected run

This bundle ships **only noise=0.0/seed_1** (the best model-performance cell) plus the random-baseline negative control. Leave these as-is unless you've generated another cell yourself.

In [ ]:
SELECTED_NOISE = 0.0
SELECTED_SEED  = 'seed_1'

# Match by numeric value against the noise_* directories on disk -- avoids float-formatting
# pitfalls (e.g. f'{0.0:g}' -> '0', but the directory is actually named 'noise_0.0').
gb1_additive_dir = PROJECT_ROOT / 'results' / 'synthetic' / 'GB1_additive'
noise_dirs = {float(p.name.split('_', 1)[1]): p for p in gb1_additive_dir.glob('noise_*') if p.is_dir()}
run_dir = noise_dirs[SELECTED_NOISE] / str(SELECTED_SEED)
linear_dir = run_dir / 'linear'

best_row = metrics_df[(metrics_df['noise'] == SELECTED_NOISE) & (metrics_df['seed'] == SELECTED_SEED)].iloc[0]
print('Selected run:', run_dir.relative_to(PROJECT_ROOT))
print(f"  model_test_spearman = {best_row['x_value']:.4f}")
for m_key in METRIC_KEYS:
    print(f'  {m_key:28s} = {best_row[m_key]:.4f}')

## 2. Load the trained model + config + official IG results

This just loads the shipped checkpoint (no retraining) and the CSV the production pipeline already wrote for this cell — both instant.

In [ ]:
model, model_cfg, norm_params = ig.load_checkpoint(run_dir, dataset_name='GB1_additive')

In [ ]:
config_snapshots = sorted(linear_dir.glob('GB1_additive_config_snapshot_*.yaml'))
import yaml
with open(config_snapshots[-1]) as f:
    run_config = yaml.safe_load(f)
print(f'Config snapshot: {config_snapshots[-1].name}')
print(yaml.dump(run_config, sort_keys=False))

In [ ]:
official_csv = linear_dir / 'GB1_additive_integrated_gradients_results.csv'
official_df = pd.read_csv(official_csv)
print('shape:', official_df.shape)
print('columns:', list(official_df.columns))
official_df.head()

Cross-check: recompute the 7 metrics for this exact cell with `compute_metrics_for_dir` and confirm they match the summary table row above (sanity check that everything lines up). We keep the full result `m` around -- it has every intermediate array (per-position |IG|, per-sequence NDCG values, category means, etc.) that Section 7 unpacks below.

In [ ]:
m = ig.compute_metrics_for_dir(linear_dir, protein='GB1', fitness_model='additive')
mutation_ndcg_mean = float(np.mean(m['mutation_ndcg_vals']))
print(f"{'metric':28s} {'summary CSV':>12s} {'recomputed':>12s}")
for key, recomputed in [('ndcg', m['ndcg']), ('auroc', m['auroc']), ('ap_graded', m['ap_graded']),
                        ('pv_ndcg_mean', m['pv_ndcg_mean']), ('sign_acc', m['sign_acc']),
                        ('cat_sep', m['cat_sep']), ('mutation_ndcg_mean_per_seq', mutation_ndcg_mean)]:
    print(f'{key:28s} {best_row[key]:12.4f} {recomputed:12.4f}')

## 3. Pick an example sequence and look at raw embeddings

First, some candidate sequences to inspect — the wildtype and a few real mutants from the test set, with their mutation count.

In [ ]:
dataset_df = pd.read_csv(DATA_DIR / 'GB1_additive.csv')
wt_seq = ig.load_wt_sequence('GB1')
print('WT sequence:', wt_seq)
print('WT length  :', len(wt_seq))

test_names = sorted(official_df[official_df['sequence_name'].str.contains(r'\[test\]', regex=True)]['sequence_name'].unique())
print(f'{len(test_names)} distinct test-split sequence entries, e.g.:')
test_names[:15]

**Pick a sequence to trace through the pipeline.** Change `SEQUENCE_NAME` to any value from the list above (or the WT itself). Defaults to the first non-WT test sequence.

In [ ]:
SEQUENCE_NAME = test_names[0]
variant_row = official_df[official_df['sequence_name'] == SEQUENCE_NAME].iloc[0]
official_seq_df = official_df[official_df['sequence_name'] == SEQUENCE_NAME].sort_values('position')
print('Chosen sequence entry:', SEQUENCE_NAME)

# Reconstruct the full variant sequence from the official CSV's per-position mutation records
variant_seq = list(wt_seq)
for _, r in official_seq_df.iterrows():
    variant_seq[int(r['position']) - 1] = r['amino_acid']
variant_seq = ''.join(variant_seq)
n_mut = sum(a != b for a, b in zip(wt_seq, variant_seq))
print(f'Variant sequence ({n_mut} mutation(s) vs WT):')
print(variant_seq)

### Step 3a — embed WT and the variant

Uses the frozen ESM-C-300M backbone (layer 33) — the first call downloads/loads the model (needs internet the very first time; a few seconds to a couple minutes depending on GPU vs CPU) and writes a small cache file under `data/embeddings/` for instant reuse afterward. **This is the tensor shape you'll see throughout: `(1, 56, 960)` = (batch, GB1 residues, ESM-C-300M embedding dim).**

In [ ]:
PLAYGROUND_DATASET_NAME = 'GB1_additive_playground'  # a small, notebook-local embedding cache
wt_emb = ig.get_sequence_embedding(wt_seq, dataset_name=PLAYGROUND_DATASET_NAME)
var_emb = ig.get_sequence_embedding(variant_seq, dataset_name=PLAYGROUND_DATASET_NAME)
print('wt_emb  shape:', wt_emb.shape, ' dtype:', wt_emb.dtype)
print('var_emb shape:', var_emb.shape, ' dtype:', var_emb.dtype)
print()
print(f'wt_emb  stats: mean={wt_emb.mean():.4f} std={wt_emb.std():.4f} min={wt_emb.min():.4f} max={wt_emb.max():.4f}')
print(f'var_emb stats: mean={var_emb.mean():.4f} std={var_emb.std():.4f} min={var_emb.min():.4f} max={var_emb.max():.4f}')
print()
print('Raw values, position 0 (first residue), first 8 embedding dims:')
print('  WT  :', wt_emb[0, 0, :8])
print('  var :', var_emb[0, 0, :8])

## 4. Hypothesis check: is the embedding space sparse?

If most of ESM-C's 960 embedding dimensions carry near-zero, redundant, or low-variance signal, that would argue for a different pooling/reduction technique than the model's plain attention-weighted average (e.g. a learned low-rank projection, or PCA-whitening before pooling). Let's check on a batch of real sequences rather than just one.

In [ ]:
# Bulk-embed a sample of sequences (one fresh ESM-C forward pass over 64 short sequences;
# a few seconds on GPU, a couple minutes on CPU -- bump N_SAMPLE up once you have a GPU kernel)
N_SAMPLE = 64
trainval_sample = dataset_df['SEQ'].astype(str).tolist()[:N_SAMPLE]
bulk_emb = ig.get_dataset_embeddings(
    trainval_sample, dataset_name='GB1_additive_playground_sparsity_sample', batch_size=64,
)
print('bulk_emb shape:', bulk_emb.shape, ' (N sequences, L positions, D dims)')

report = ig.embedding_sparsity_report(bulk_emb)
print(f"mean fraction near-zero per dim : {report['mean_frac_near_zero_per_dim']:.4f}")
print(f"max  fraction near-zero per dim : {report['max_frac_near_zero_per_dim']:.4f}")
print(f"dims needed for 50% variance    : {report['n_dims_for_50pct_variance']} / {report['d_embedding']}")
print(f"dims needed for 90% variance    : {report['n_dims_for_90pct_variance']} / {report['d_embedding']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.cumsum(report['explained_var_ratio'][:200]))
axes[0].axhline(0.9, color='grey', ls='--', lw=1)
axes[0].set_xlabel('# principal components'); axes[0].set_ylabel('cumulative explained variance')
axes[0].set_title('PCA of pooled-position embeddings')

axes[1].hist(report['frac_near_zero_per_dim'], bins=40, color='#4C72B0')
axes[1].set_xlabel('fraction of values near zero, per dimension'); axes[1].set_ylabel('# dims')
axes[1].set_title('Per-dimension sparsity')
fig.tight_layout(); plt.show()

**Interpretation space** — if the left plot saturates well before D=960 (e.g. 90% of variance in &lt;100 components), the model's attention-weighted average is working in an effectively low-dimensional space, and a PCA/learned-projection pooling head might recover the same signal with far fewer parameters (or expose which components actually drive predictions). If it's nearly a straight diagonal line, the space is closer to full-rank and this hypothesis is weaker.

## 5. Normalize, forward pass, and attention-pooling weights

Two things happen before the MLP: (1) the raw embedding is z-score normalised using the *training-set* mean/scale saved in the checkpoint, (2) the attention-pooling layer collapses `(1, 56, 960)` down to one 960-dim vector via a learned, position-wise softmax.

In [ ]:
mean, scale = norm_params['mean'], norm_params['scale']
var_emb_t = torch.tensor(var_emb, dtype=torch.float32, device=ig.DEVICE)
norm_emb = ig.normalize_embedding(var_emb_t, mean, scale)
print('normalized embedding shape:', tuple(norm_emb.shape))
print(f'post-normalization stats: mean={norm_emb.mean().item():.4f} std={norm_emb.std().item():.4f}')

pred = ig.model_forward(model, norm_emb, model_cfg['pooling_method'])
print(f'model prediction (rescaled fitness) shape: {tuple(pred.shape)}, value = {pred.item():.4f}')
print(f"actual fitness (from CSV)                 = {variant_row['actual_activity']:.4f}")

In [ ]:
attn_weights = ig.get_attention_pooling_weights(model, norm_emb)[0].cpu().numpy()  # (L,)
print('attention weights shape:', attn_weights.shape, ' sum =', attn_weights.sum())

iv, ann_mask, by_cat, bg_positions = ig.load_ground_truth('GB1')
ann_colors = ['#E63946' if ann_mask[i] else '#AAB2BD' for i in range(len(ann_mask))]
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(1, len(attn_weights) + 1), attn_weights, color=ann_colors)
ax.set_xlabel('residue position (1-indexed)'); ax.set_ylabel('attention weight')
ax.set_title(f'Attention-pooling weights for {SEQUENCE_NAME}  (red = biologically annotated position)')
fig.tight_layout(); plt.show()

## 6. Integrated gradients, step by step

This is the core of IGLOO: walk a linear path in embedding space from the WT baseline to the variant, accumulate gradients of the model's prediction with respect to the embedding at each step (trapezoidal rule), and scale by the total displacement. `steps=500` matches the production config exactly; try dropping it to see how attribution stability degrades with fewer steps.

In [ ]:
STEPS = 500       # production default; try e.g. 10, 50 to see the effect of coarser integration
BATCH_SIZE = 32

ig_result = ig.compute_linear_ig_stepwise(
    model, input_emb=var_emb, baseline_emb=wt_emb,
    mean=mean, scale=scale, pooling_method=model_cfg['pooling_method'],
    steps=STEPS, batch_size=BATCH_SIZE, verbose=True,
)
print()
print(f"completeness check (should be ~0): sum(signed_ig) - (F(var)-F(wt)) = {ig_result['completeness_gap']:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(1, len(ig_result['signed_ig']) + 1), ig_result['signed_ig'], color=ann_colors)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('residue position'); ax.set_ylabel('signed integrated gradient')
ax.set_title(f'Per-position signed IG for {SEQUENCE_NAME}  ({STEPS} steps, linear path)')
fig.tight_layout(); plt.show()

### Cross-check against the official (production-computed) values
This reimplementation should closely match what's already in the results CSV for this exact sequence — small floating-point differences are expected, large ones would flag a bug.

In [ ]:
official_ig_lookup = ig.load_official_ig_for_sequence(official_csv, SEQUENCE_NAME)
mutated_positions = official_ig_lookup['position'].values.astype(int) - 1   # 0-indexed
official_vals = official_ig_lookup['integrated_gradient'].values
notebook_vals = ig_result['signed_ig'][mutated_positions]

cmp_df = pd.DataFrame({
    'position': official_ig_lookup['position'].values,
    'official_csv_ig': official_vals,
    'notebook_recomputed_ig': notebook_vals,
    'abs_diff': np.abs(official_vals - notebook_vals),
})
print(f"max abs diff: {cmp_df['abs_diff'].max():.6f}   Pearson r: {np.corrcoef(official_vals, notebook_vals)[0,1]:.6f}")
cmp_df

## 7. All 7 metrics — intermediate steps, for the selected (best) model

Every metric below is built from the same underlying ingredients you've already seen: the per-position |IG| values, the ground-truth importance annotation, and per-sequence quantities computed the same way as `ig_result` above but pooled/averaged across the whole test set (via `m = compute_metrics_for_dir(...)` from Section 2). This section unpacks each of the 7 metrics' intermediate arrays so you can see exactly what goes into the final number — not just the number itself.

### 7a. Global ranking family — NDCG, AUROC, AP@R

In [ ]:
print('Global |IG| vector: mean |IG| at each position, pooled across every test-set sequence.')
print('  m["ig_vec"] shape:', m['ig_vec'].shape)
print('  ground-truth importance vector (iv) shape:', iv.shape, ' annotated positions:', int(ann_mask.sum()), '/', len(ann_mask))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].bar(range(1, 57), m['ig_vec'], color=ann_colors)
axes[0].set_ylabel('mean |IG|\n(all sequences)')
axes[0].set_title('What the model attributes importance to')
axes[1].bar(range(1, 57), iv, color=ann_colors)
axes[1].set_ylabel('ground-truth\nimportance weight')
axes[1].set_xlabel('residue position'); axes[1].set_title('What the ground-truth annotation says is actually important')
fig.tight_layout(); plt.show()

print(f"NDCG     = {m['ndcg']:.4f}  -- ranks all 56 positions by |IG|, rewards matching the ground-truth ranking (top-heavy, log-discounted)")
print(f"AUROC    = {m['auroc']:.4f}  -- binary: can |IG| alone separate ANY annotated position from background? (0.5 = chance)")
print(f"AP@R     = {m['ap_graded']:.4f}  -- graded average precision: rewards retrieving HIGH-importance positions before LOW-importance ones")

### 7b. Per-variant family — per-variant NDCG, sign accuracy

In [ ]:
print(f"Per-variant NDCG: computed separately for each of {len(m['pv_ndcg_vals'])} test sequences, then averaged.")
print(f"  mean = {m['pv_ndcg_mean']:.4f}   std = {m['pv_ndcg_std']:.4f}")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(m['pv_ndcg_vals'], bins=40, color='#4C72B0')
ax.axvline(m['pv_ndcg_mean'], color='#E63946', ls='--', label=f"mean={m['pv_ndcg_mean']:.3f}")
ax.set_xlabel('per-sequence NDCG'); ax.set_ylabel('# sequences'); ax.legend()
ax.set_title('Distribution of per-variant NDCG across all test-set sequences')
fig.tight_layout(); plt.show()

In [ ]:
print(f'Sign accuracy demo, for {SEQUENCE_NAME}:')
seq_signed_ig = official_ig_lookup['integrated_gradient'].values
threshold = float(np.median(np.abs(seq_signed_ig)))
print(f'  per-sequence threshold = median(|signed IG|) = {threshold:.4f}')
any_checked = False
for _, r in official_seq_df.iterrows():
    pos0 = int(r['position']) - 1
    is_annotated = bool(ann_mask[pos0])
    is_mutated = r['amino_acid'] != wt_seq[pos0]
    if is_annotated and is_mutated:
        any_checked = True
        passed = r['integrated_gradient'] < -threshold
        print(f"  position {r['position']} ({wt_seq[pos0]}->{r['amino_acid']}): signed IG={r['integrated_gradient']:.4f}"
              f"  < -threshold({-threshold:.4f})? {'YES -> counted correct' if passed else 'NO -> counted wrong'}")
if not any_checked:
    print('  (this sequence has no mutations at annotated positions -- pick another SEQUENCE_NAME in Section 3 to see this check fire)')
print(f"\nOverall sign accuracy (ALL sequences, ALL mutated annotated positions) = {m['sign_acc']:.4f}")

### 7c. Category-level — category separation ratio

In [ ]:
CATS = ['background', 'folding', 'structural', 'binding_site']
CAT_COLORS = ['#AAB2BD', '#F4A261', '#2A9D8F', '#E63946']
cat_vals = [m['cat_means'].get(c, float('nan')) for c in CATS]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(CATS, cat_vals, color=CAT_COLORS)
ax.set_ylabel('mean signed IG')
ax.set_title('Category mean signed IG\n(expected order: background < folding < structural < binding_site)')
fig.tight_layout(); plt.show()

top_cat = max((c for c in CATS if c != 'background'), key=lambda c: m['cat_means'][c])
print(f"Top non-background category = {top_cat}  (mean={m['cat_means'][top_cat]:.4f})")
print(f"background mean = {m['cat_means']['background']:.4f}")
print(f"cat_sep = top / background = {m['cat_sep']:.4f}")

### 7d. Mutation-level — per-sequence mutation NDCG

In [ ]:
print(f"Mutation-level NDCG: computed per-sequence (>=2 mutations) via leave-one-out (LOO)")
print(f"fitness deltas as ground truth, ranked against |IG| -- {len(m['mutation_ndcg_vals'])} sequences.")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(m['mutation_ndcg_vals'], bins=30, color='#2A9D8F')
ax.set_xlabel('per-sequence mutation NDCG'); ax.set_ylabel('# sequences')
ax.set_title('Distribution of per-sequence mutation NDCG')
fig.tight_layout(); plt.show()

In [ ]:
# Hands-on LOO demo for a real multi-mutant sequence (';'-separated = 2+ mutations)
multi_mut_names = [n for n in test_names if ';' in n]
demo_name = multi_mut_names[0] if multi_mut_names else SEQUENCE_NAME
demo_rows = official_df[official_df['sequence_name'] == demo_name].sort_values('position')
demo_mut_list = [
    (int(r.position) - 1, wt_seq[int(r.position) - 1], r.amino_acid)
    for _, r in demo_rows.iterrows() if r.amino_acid != wt_seq[int(r.position) - 1]
]
print(f'Demo sequence: {demo_name}  ({len(demo_mut_list)} mutations)')

loo_contribs = ig.compute_loo_contributions('GB1', 'additive', wt_seq, demo_mut_list)
demo_ig_lookup = {int(row.position): row.integrated_gradient for _, row in demo_rows.iterrows()}
demo_cmp = pd.DataFrame({
    'position': [p + 1 for p, _, _ in demo_mut_list],
    'mutation': [f'{a}{p+1}{b}' for p, a, b in demo_mut_list],
    'LOO_fitness_delta (ground truth relevance)': loo_contribs,
    'IG_signed': [demo_ig_lookup[p + 1] for p, _, _ in demo_mut_list],
    'IG_abs': [abs(demo_ig_lookup[p + 1]) for p, _, _ in demo_mut_list],
})
print('Positive LOO delta = removing that mutation INCREASED fitness = that mutation was damaging.')
print('Mutation NDCG for this sequence ranks IG_abs against clip(LOO_fitness_delta, 0, inf).')
demo_cmp

## 8. Compare with a "complete random" model (shuffled labels)

To know whether any of the numbers above reflect real learned signal, we need a negative control: a model trained on the exact same architecture/data but with fitness labels fully shuffled across sequences before training, so there is no real sequence-fitness relationship left to learn. Its `model_test_spearman` should be ~0 (confirmed below). Comparing its internals to the best model's, for the *same* example sequence, shows what "a model that learned nothing" actually looks like at every stage of the pipeline.

(Note: the config snapshot for this run doesn't display `shuffle_labels: true` -- a cosmetic gap in how the snapshot is written, not a bug in the run itself. The near-zero `model_test_spearman` below is the real, direct confirmation that labels were shuffled.)

In [ ]:
random_run_dir = PROJECT_ROOT / 'results' / 'synthetic' / 'GB1_additive' / 'random_baseline' / 'seed_0'
random_linear_dir = random_run_dir / 'linear'
random_model, random_model_cfg, random_norm_params = ig.load_checkpoint(random_run_dir, dataset_name='GB1_additive')

In [ ]:
m_random = ig.compute_metrics_for_dir(random_linear_dir, protein='GB1', fitness_model='additive')
print(f"best model   model_test_spearman = {m['model_test_spearman']:.4f}")
print(f"random model model_test_spearman = {m_random['model_test_spearman']:.4f}   (near 0 = confirms no real signal was learned)")

### Embeddings are identical either way

Worth confirming directly: `wt_emb`/`var_emb` from Section 3 don't depend on which downstream model consumes them (the frozen ESM-C backbone doesn't know or care what the prediction head was trained on) -- only what happens *after* embedding differs.

In [ ]:
random_norm_emb = ig.normalize_embedding(var_emb_t, random_norm_params['mean'], random_norm_params['scale'])
random_pred = ig.model_forward(random_model, random_norm_emb, random_model_cfg['pooling_method'])

print(f"best model prediction   : {pred.item():.4f}")
print(f"random model prediction : {random_pred.item():.4f}")
print(f"actual fitness          : {variant_row['actual_activity']:.4f}")

### Attention-pooling weights: best model vs. random model

In [ ]:
random_attn = ig.get_attention_pooling_weights(random_model, random_norm_emb)[0].cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].bar(range(1, 57), attn_weights, color=ann_colors)
axes[0].set_title(f'Best model attention weights  (model_test_spearman={m["model_test_spearman"]:.3f})')
axes[1].bar(range(1, 57), random_attn, color=ann_colors)
axes[1].set_title(f'Random-shuffled-label model attention weights  (model_test_spearman={m_random["model_test_spearman"]:.3f})')
axes[1].set_xlabel('residue position')
fig.tight_layout(); plt.show()

### Full integrated gradients: best model vs. random model, same sequence

In [ ]:
random_ig_result = ig.compute_linear_ig_stepwise(
    random_model, input_emb=var_emb, baseline_emb=wt_emb,
    mean=random_norm_params['mean'], scale=random_norm_params['scale'],
    pooling_method=random_model_cfg['pooling_method'],
    steps=STEPS, batch_size=BATCH_SIZE, verbose=False,
)
corr = np.corrcoef(ig_result['signed_ig'], random_ig_result['signed_ig'])[0, 1]
print(f'Pearson correlation between best-model IG and random-model IG, same sequence: {corr:.4f}')
print('(low/near-zero correlation would show the two models attribute importance completely differently;')
print(' a high correlation would be a red flag that the metric or attribution method is not sensitive to what the model actually learned)')

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].bar(range(1, 57), ig_result['signed_ig'], color=ann_colors)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Best model: signed IG')
axes[1].bar(range(1, 57), random_ig_result['signed_ig'], color=ann_colors)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Random model: signed IG')
axes[1].set_xlabel('residue position')
fig.tight_layout(); plt.show()

### Pooled |IG| vector — best vs. random model (the values behind MAE/R²)

Section 7a plotted `m['ig_vec']` (mean |IG| per position, pooled across every test-set sequence) for the best model only, against the ground truth. Here it is for **both** models side by side, plus the exact z-scored versions and the resulting MAE/R² -- i.e. the actual values that feed the MAE(z-scored)/R²(z-scored) metrics discussed above.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

print('Pooled |IG| vector: mean |IG| at each position, pooled across every test-set sequence.')
print(f"  best model:   {len(m['pv_ndcg_vals'])} test sequences pooled,   ig_vec shape {m['ig_vec'].shape}")
print(f"  random model: {len(m_random['pv_ndcg_vals'])} test sequences pooled,   ig_vec shape {m_random['ig_vec'].shape}")
print()

z_best   = (m['ig_vec']        - m['ig_vec'].mean())        / m['ig_vec'].std()
z_random = (m_random['ig_vec'] - m_random['ig_vec'].mean()) / m_random['ig_vec'].std()
z_iv     = (iv                 - iv.mean())                 / iv.std()

mae_best,   r2_best   = mean_absolute_error(z_iv, z_best),   r2_score(z_iv, z_best)
mae_random, r2_random = mean_absolute_error(z_iv, z_random), r2_score(z_iv, z_random)

print(f"Best model:   MAE(z-scored |IG|, z-scored ground truth) = {mae_best:.4f}   R²(identity line) = {r2_best:.4f}")
print(f"Random model: MAE(z-scored |IG|, z-scored ground truth) = {mae_random:.4f}   R²(identity line) = {r2_random:.4f}")
print("(R² here works out to exactly 2*Pearson_r - 1 once both sides are z-scored -- it is not new")
print(" information beyond correlation; see the discussion above for why.)")

# Raw pooled |IG| vectors -- best and random share a Y-axis (both are |IG|, so
# the magnitude gap is directly visible); ground truth is on its own axis
# (different, incommensurable units/scale, so sharing would not be meaningful).
ig_ylim = (0, max(m['ig_vec'].max(), m_random['ig_vec'].max()) * 1.08)
fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)
axes[0].bar(range(1, 57), m['ig_vec'], color=ann_colors); axes[0].set_ylim(ig_ylim)
axes[0].set_ylabel('mean |IG|'); axes[0].set_title('Best model: pooled mean |IG| across all test sequences')
axes[1].bar(range(1, 57), m_random['ig_vec'], color=ann_colors); axes[1].set_ylim(ig_ylim)
axes[1].set_ylabel('mean |IG|'); axes[1].set_title('Random model: pooled mean |IG| across all test sequences (same Y-axis as above)')
axes[2].bar(range(1, 57), iv, color=ann_colors)
axes[2].set_ylabel('importance\nweight'); axes[2].set_xlabel('residue position')
axes[2].set_title('Ground-truth importance weight (different scale -- not directly comparable in raw units)')
fig.tight_layout(); plt.show()

# What MAE/R2 actually compare: all three z-scored onto the same standardized scale.
fig2, ax2 = plt.subplots(figsize=(11, 4))
x = np.arange(1, 57)
ax2.bar(x - 0.2, z_best, width=0.4, color='#2A9D8F', label='best model |IG| (z-scored)')
ax2.bar(x + 0.2, z_random, width=0.4, color='#E63946', alpha=0.75, label='random model |IG| (z-scored)')
ax2.plot(x, z_iv, color='black', lw=1.3, marker='o', ms=3, label='ground truth (z-scored)')
ax2.axhline(0, color='grey', lw=0.6)
ax2.set_xlabel('residue position'); ax2.set_ylabel('z-score')
ax2.legend(fontsize=8, loc='upper right')
ax2.set_title('Z-scored |IG| (best vs. random) vs. z-scored ground truth -- exactly what MAE/R² compare')
fig2.tight_layout(); plt.show()

### Same Y-axis: attention weights and signed IG, best vs. random model

The two comparisons earlier in this section (attention weights, and signed IG) each plot best vs. random with independently auto-scaled Y-axes -- which hides how much smaller the random model's raw values actually are (its panel just stretches to fill the same visual height as the best model's, even though the real numbers are roughly two to three orders of magnitude smaller). Re-plotting both pairs with a shared, matched Y-axis makes the actual magnitude gap directly visible instead of hidden by auto-scaling.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
axes[0].bar(range(1, 57), attn_weights, color=ann_colors)
axes[0].set_title(f'Best model attention weights  (model_test_spearman={m["model_test_spearman"]:.3f})')
axes[0].set_ylabel('attention weight')
axes[1].bar(range(1, 57), random_attn, color=ann_colors)
axes[1].set_title(f'Random-shuffled-label model attention weights  (model_test_spearman={m_random["model_test_spearman"]:.3f})')
axes[1].set_xlabel('residue position'); axes[1].set_ylabel('attention weight')
fig.suptitle('Same Y-axis on both panels -- magnitude is now directly comparable', fontsize=10, color='#555', y=1.02)
fig.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
axes[0].bar(range(1, 57), ig_result['signed_ig'], color=ann_colors)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Best model: signed IG'); axes[0].set_ylabel('signed IG')
axes[1].bar(range(1, 57), random_ig_result['signed_ig'], color=ann_colors)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Random model: signed IG')
axes[1].set_xlabel('residue position'); axes[1].set_ylabel('signed IG')
fig.suptitle('Same Y-axis on both panels -- magnitude is now directly comparable', fontsize=10, color='#555', y=1.02)
fig.tight_layout(); plt.show()

## 9. Floor diagnostic — why can't some metrics distinguish real from random?

Side-by-side: the best model's score vs. the random model's score, for all 7 metrics. A metric that's actually measuring learned signal should show a large gap; a metric near-equal for both models can't be trusted to tell you whether attribution quality is real.

In [ ]:
floor_rows = []
mutation_ndcg_mean_random = float(np.mean(m_random['mutation_ndcg_vals']))
for key in METRIC_KEYS:
    best_v = mutation_ndcg_mean if key == 'mutation_ndcg_mean_per_seq' else m[key]
    rand_v = mutation_ndcg_mean_random if key == 'mutation_ndcg_mean_per_seq' else m_random[key]
    floor_rows.append({'metric': METRIC_LABELS[key], 'best_model': best_v, 'random_model': rand_v, 'gap': best_v - rand_v})
floor_df = pd.DataFrame(floor_rows)
floor_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(floor_df))
ax.barh(y_pos - 0.2, floor_df['best_model'], height=0.4, color='#2A9D8F', label='best model')
ax.barh(y_pos + 0.2, floor_df['random_model'], height=0.4, color='#E63946', label='random model (shuffled labels)')
ax.set_yticks(y_pos); ax.set_yticklabels(floor_df['metric'])
ax.set_xlabel('raw metric value (note: different metrics have different natural scales)')
ax.set_title('Best model vs. random-shuffled-label model, one cell each')
ax.legend(); ax.invert_yaxis()
fig.tight_layout(); plt.show()

### Same picture, but across the *entire* noise-level sweep

Using the precomputed 60-cell sweep (`metrics_df`) and the 5-seed random baseline (`baseline_df`) from Section 1 -- a compact overview of all 7 metrics at once, each panel showing the per-noise-level mean (5 seeds) as a line, with the random baseline's mean ± 95% CI as a shaded horizontal band. Metrics whose real-model line stays inside the random band can't distinguish noise level from chance at all.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, key in enumerate(METRIC_KEYS):
    ax = axes[i]
    linear_df = metrics_df[metrics_df['interp'] == 'linear']
    agg = linear_df.groupby('noise')[key].mean().reset_index().sort_values('noise')
    ax.plot(agg['noise'], agg[key], 'o-', color='#2A9D8F', label='real model (mean of 5 seeds)')
    rand_vals = baseline_df[key].values
    rand_mean, rand_std = rand_vals.mean(), rand_vals.std(ddof=1)
    from scipy import stats as _stats
    rand_ci = _stats.t.ppf(0.975, df=len(rand_vals)-1) * rand_std / np.sqrt(len(rand_vals))
    ax.axhspan(rand_mean - rand_ci, rand_mean + rand_ci, color='#E63946', alpha=0.15)
    ax.axhline(rand_mean, color='#E63946', ls='--', lw=1, label='random baseline mean ± 95% CI')
    ax.set_title(METRIC_LABELS[key], fontsize=10)
    ax.set_xlabel('noise level', fontsize=8)
axes[0].legend(fontsize=7, loc='best')
axes[-1].axis('off')
fig.suptitle('All 7 metrics vs. noise level, with the random-baseline floor', y=1.02)
fig.tight_layout(); plt.show()

## 10. Prototype a new metric — calibrating against the random floor

One concrete fix for the "can't tell real from random" problem: instead of reporting a metric's raw value, report how many standard deviations above the random-baseline distribution it sits. This puts every metric on a comparable scale (a z-score) and directly answers "is this better than chance, and by how much" -- which the raw values above don't do on their own (e.g. NDCG=0.90 sounds great in isolation, but so does the random baseline's own ~0.89).

In [ ]:
def zscore_vs_random(value, random_seed_values):
    """How many random-baseline standard deviations above the random mean is `value`?"""
    random_seed_values = np.asarray(random_seed_values)
    return (value - random_seed_values.mean()) / random_seed_values.std(ddof=1)

print(f"{'metric':28s} {'raw value':>10s} {'z-score vs random':>20s}")
z_rows = []
for key in METRIC_KEYS:
    best_v = best_row[key]
    z = zscore_vs_random(best_v, baseline_df[key].values)
    z_rows.append({'metric': METRIC_LABELS[key], 'raw_value': best_v, 'z_score_vs_random': z})
    print(f'{key:28s} {best_v:10.4f} {z:20.2f}')
pd.DataFrame(z_rows)

**Try your own metric here.** Any function that takes the raw arrays already sitting in `m` (e.g. `m['ig_vec']`, `m['pv_ndcg_vals']`, `m['cat_means']`, `m['mutation_ndcg_vals']`) and the ground truth (`iv`, `ann_mask`, `by_cat`) can be tested the same way: compute it for the best model, compute it for all 5 random-baseline seeds (loop over `results/synthetic/GB1_additive/random_baseline/seed_{0..4}/linear/`, only seed_0 is bundled here so you'd need to regenerate the others -- see the README), then z-score it. A genuinely better metric should show a bigger z-score than the 7 above for the same underlying data.

## 11. Recreate the metric-vs-model-performance plots, for all 7 metrics

Same plots as in the original analysis, rendered directly here (no file written) -- one per metric, each with the full per-noise-level mean ± 95% CI, the random-baseline overlay, and an extrapolated "noise level where the model becomes indistinguishable from random" estimate.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.plot_ndcg_vs_model_performance import plot_one_interp, Y_METRIC_LABELS
%matplotlib inline
# ^ the import above sets matplotlib.use('Agg') when the module is run as a CLI
# script; harmless there, but it also fires on import, so we reassert the
# inline backend here or the figures below would silently fail to display.

for y_metric in METRIC_KEYS:
    print(f'\n=== {METRIC_LABELS[y_metric]} ({y_metric}) ===')
    plot_one_interp(
        metrics_df.copy(), interp='linear', x_metric='model_test_spearman', y_metric=y_metric,
        output_path=None, dpi=100, baseline_df=baseline_df, show=True,
    )